# 모델 평가와 Random Forest

## Bike Sharing 예측모형의 validation과 최종 평가

같은 평가 표본에서 평균 baseline, Linear Regression, Decision Tree와 Random Forest의 오류를 비교하고 큰 잔차가 발생한 관측을 확인합니다.


## 이번 실습에서 완성할 것

- 네 후보의 fit·validation 지표를 같은 행에서 비교합니다.
- Random Forest의 개별 tree 평균을 확인합니다.
- 깊이와 validation seed 민감성, 선택모형의 최종 test 오류 조건을 점검합니다.

**완료 결과:** 평가표, 큰 오류 조건표, 주지표 선택 이유와 아직 검증하지 않은 범위


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI Steel Industry Energy Consumption

- **규모와 grain:** 35,040개의 15분 전력사용 관측
- **target:** `Usage_kWh`
- **비교모형:** 평균 baseline, Linear Regression, Decision Tree, Random Forest
- **오류를 해석할 조건:** 시간, 요일, 부하유형, power factor

5교시와 같은 자료를 사용합니다.

동일한 fit/validation 행을 네 후보에 사용하고, 선택을 마친 Pipeline 하나만 test에 적용하며, MAE·RMSE·R²뿐 아니라 가장 큰 잔차가 발생한 운전조건을 확인합니다.

### 적용 실습: UCI Bike Sharing

시간별 대여건수 `cnt`를 같은 네 모형으로 예측합니다. `cnt`의 범위는 1~977건이며 시간대와 근무일에 따라 분포가 크게 달라질 수 있습니다. 평균점수만 보고하지 않고 큰 오류가 발생한 날짜·시간·날씨 조건을 함께 조사합니다.
> `casual`과 `registered`는 누수 방지를 위해 계속 제외합니다.


## 이번 실습의 분석 순서

점수를 보기 전에 주지표를 `MAE` 또는 `RMSE`로 정합니다. 네 후보는 같은 fit·validation 행에서 비교하고, 선택규칙과 선택근거를 기록한 뒤 최종 절에서만 test를 한 번 평가합니다.

**남길 결과:** fit–validation 비교표, 선택규칙, 최종 test 지표표, 큰 오류 관측과 조건 요약


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | 네 후보의 fit–validation 비교와 마지막 test·오류분석·PASS |
| **시간이 남으면** | Random Forest 깊이 validation |
| **수업 후 확장(선택)** | validation seed 안정성 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 평가 표본과 비교조건

후보 비교와 최종 평가의 자료 역할을 분리합니다.

| 단계 | 사용하는 행 | 목적 |
|---|---|---|
| fit | train 내부의 학습 행 | 전처리와 모형 적합 |
| validation | train 내부의 비교 행 | 주지표에 따른 후보·설정 선택 |
| test | 처음에 보관한 평가 행 | 선택된 Pipeline의 마지막 1회 평가 |

모든 후보는 같은 fit·validation 행과 같은 지표를 사용합니다. test에는 validation에서 정한 Pipeline 하나만 적용합니다. 분포표에서는 세 자료의 행 수, 평균, 표준편차와 범위를 확인합니다.



In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

data = pd.read_csv(find_course_data("bike/hour.csv"))
target_name = "cnt"
numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
analysis_name = "Bike model validity"

feature_columns = numeric_features + categorical_features
X = data[feature_columns]
y = data[target_name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_selection_fit, X_selection_validation, y_selection_fit, y_selection_validation = train_test_split(
    X_train, y_train, test_size=0.20, random_state=43
)
PRIMARY_METRIC = "MAE"  # 평균 오류를 중시하면 MAE, 큰 오류를 더 중시하면 RMSE
if PRIMARY_METRIC not in {"MAE", "RMSE"}:
    raise ValueError("PRIMARY_METRIC은 MAE 또는 RMSE여야 합니다.")
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
print("fit/validation/test:", X_selection_fit.shape, X_selection_validation.shape, X_test.shape)


split_summary = pd.DataFrame({
    "fit": y_selection_fit.describe(),
    "validation": y_selection_validation.describe(),
    "test": y_test.describe(),
}).round(2)
print("\n[fit·validation·test target 요약]")
display(split_summary)


## 2. 세 지표와 fit–validation 격차

MAE는 평균 절대오차, RMSE는 큰 오차에 더 민감하며 R²는 test 실제값의 평균을 반복한 예측과 비교해 제곱오차가 얼마나 줄었는지를 나타냅니다. train 성능도 함께 기록해 과적합을 찾습니다.


In [ ]:
models = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "linear regression": Pipeline([
        ("prep", clone(preprocess)),
        ("model", LinearRegression()),
    ]),
    "decision tree": Pipeline([
        ("prep", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=2,
            random_state=42,
        )),
    ]),
    "random forest": Pipeline([
        ("prep", clone(preprocess)),
        ("model", RandomForestRegressor(
            n_estimators=60,
            max_depth=14,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

rows = []
fitted_candidates = {}
validation_predictions = {}
for name, model in models.items():
    candidate = clone(model)
    candidate.fit(X_selection_fit, y_selection_fit)
    fitted_candidates[name] = candidate
    for split_name, split_X, split_y in [
        ("fit", X_selection_fit, y_selection_fit),
        ("validation", X_selection_validation, y_selection_validation),
    ]:
        prediction = candidate.predict(split_X)
        rows.append({
            "model": name,
            "split": split_name,
            "MAE": mean_absolute_error(split_y, prediction),
            "RMSE": root_mean_squared_error(split_y, prediction),
            "R2": r2_score(split_y, prediction),
        })
        if split_name == "validation":
            validation_predictions[name] = prediction

result_table = pd.DataFrame(rows)
selection_table = (
    result_table.loc[result_table["split"].eq("validation"), [
        "model", "MAE", "RMSE", "R2"
    ]]
    .sort_values(PRIMARY_METRIC)
    .reset_index(drop=True)
)
selection_rule = f"validation {PRIMARY_METRIC}가 가장 작은 후보"
metric_leader_name = selection_table.loc[0, "model"]
selected_model_name = metric_leader_name

comparison = result_table.pivot(
    index="model", columns="split", values=["MAE", "RMSE", "R2"]
)
comparison.columns = [f"{metric}_{split}" for metric, split in comparison.columns]
comparison[f"{PRIMARY_METRIC}_validation_minus_fit"] = (
    comparison[f"{PRIMARY_METRIC}_validation"]
    - comparison[f"{PRIMARY_METRIC}_fit"]
)

print(f"[fit과 validation에서 네 후보 비교 — 주지표: {PRIMARY_METRIC}]")
display(comparison.round(3).sort_values(f"{PRIMARY_METRIC}_validation"))
print("이 실습에서 사전에 정한 선택 규칙:", selection_rule)
print("선택 규칙에 따른 1차 후보:", selected_model_name)
print("아직 test는 예측하지 않았습니다. 깊이·seed 검토 뒤 마지막 절에서 한 번 평가합니다.")


## 3. validation에서 1차 후보 진단

사전에 정한 주지표로 1차 후보를 고른 뒤, fit–validation 격차와 validation의 큰 오류 관측을 확인합니다. 이 단계의 목적은 선택규칙과 진단근거를 분리해 기록하는 것이며, 아직 test는 사용하지 않습니다.


In [ ]:
metric_reference_name = selected_model_name
selected_fit_value = result_table.query(
    "model == @selected_model_name and split == 'fit'"
)[PRIMARY_METRIC].iloc[0]
selected_validation_value = result_table.query(
    "model == @selected_model_name and split == 'validation'"
)[PRIMARY_METRIC].iloc[0]
selected_generalization_gap = selected_validation_value - selected_fit_value

validation_error_table = X_selection_validation.copy()
validation_error_table["observed"] = y_selection_validation
validation_error_table["predicted"] = validation_predictions[selected_model_name]
validation_error_table["residual"] = (
    validation_error_table["observed"] - validation_error_table["predicted"]
)
validation_error_table["absolute_error"] = validation_error_table["residual"].abs()

print(f"[선택 규칙에 따른 1차 후보: {selected_model_name}]")
display(selection_table.round(3))
print(
    f"{PRIMARY_METRIC}의 validation - fit 격차:",
    round(float(selected_generalization_gap), 3),
)
print("[validation에서 절대오차가 큰 관측 8건 — test 결과가 아님]")
validation_condition_columns = [column for column in [
    "hr", "workingday", "weathersit", "temp", "hum"
] if column in validation_error_table.columns]
display(validation_error_table.nlargest(8, "absolute_error")[[
    "observed", "predicted", "residual", "absolute_error",
    *validation_condition_columns,
]].round(2))


## 시간이 남으면 — Random Forest 깊이 validation

`max_depth`만 4·14·제한 없음으로 바꾸고 같은 fit·validation 행에서 MAE와 RMSE를 비교합니다. tree 수, 전처리, 모형 seed와 비교행은 그대로 유지하며 test는 사용하지 않습니다.

결과를 보기 전에 깊이가 커질수록 fit 오차와 validation 오차가 각각 어떻게 변할지 예상하고, 실행 뒤에는 주지표와 `validation - fit` 격차를 함께 기록합니다.


In [ ]:
# 수업 중 시간이 남으면 실행합니다. test는 사용하지 않습니다.
FOREST_DEPTHS = [4, 14, None]
depth_rows = []
for depth in FOREST_DEPTHS:
    candidate = Pipeline([
        ("prep", clone(preprocess)),
        ("model", RandomForestRegressor(
            n_estimators=60,
            max_depth=depth,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ])
    candidate.fit(X_selection_fit, y_selection_fit)
    fit_prediction = candidate.predict(X_selection_fit)
    validation_prediction = candidate.predict(X_selection_validation)
    depth_rows.append({
        "depth_label": "None (제한 없음)" if depth is None else str(depth),
        "fit_MAE": mean_absolute_error(y_selection_fit, fit_prediction),
        "validation_MAE": mean_absolute_error(y_selection_validation, validation_prediction),
        "fit_RMSE": root_mean_squared_error(y_selection_fit, fit_prediction),
        "validation_RMSE": root_mean_squared_error(
            y_selection_validation, validation_prediction
        ),
    })

depth_experiment = pd.DataFrame(depth_rows)
validation_metric_column = f"validation_{PRIMARY_METRIC}"
gap_column = f"validation_minus_fit_{PRIMARY_METRIC}"
depth_experiment[gap_column] = (
    depth_experiment[validation_metric_column]
    - depth_experiment[f"fit_{PRIMARY_METRIC}"]
)
depth_experiment = depth_experiment.sort_values(validation_metric_column).reset_index(drop=True)
selected_forest_depth_label = depth_experiment.loc[0, "depth_label"]
selected_forest_depth = (
    None if selected_forest_depth_label.startswith("None")
    else int(selected_forest_depth_label)
)

print(f"[test를 사용하지 않은 Random Forest 깊이 비교 — {PRIMARY_METRIC}]")
print("fit/validation 행, tree 수, 전처리와 seed는 같고 max_depth만 다릅니다.")
display(depth_experiment.round(3))
print("validation 주지표가 가장 작은 깊이:", selected_forest_depth_label)


## 수업 후 확장(선택) — validation 분할 seed 안정성

한 번의 validation 점수는 선택된 행에 따라 달라질 수 있습니다. Random Forest의 나머지 조건을 고정하고, train 내부 validation 분할 seed만 7·42·99로 바꾸어 MAE와 RMSE의 평균과 범위를 확인합니다. 이 절에서도 test는 사용하지 않습니다.


In [ ]:
# 수업 후 확장입니다. test는 사용하지 않습니다.
depth_for_stability = globals().get("selected_forest_depth", 14)
stability_rows = []
for validation_seed in [7, 42, 99]:
    X_stability_fit, X_stability_validation, y_stability_fit, y_stability_validation = train_test_split(
        X_train, y_train, test_size=0.20, random_state=validation_seed
    )
    stability_model = Pipeline([
        ("prep", clone(preprocess)),
        ("model", RandomForestRegressor(
            n_estimators=60,
            max_depth=depth_for_stability,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )),
    ])
    stability_model.fit(X_stability_fit, y_stability_fit)
    stability_prediction = stability_model.predict(X_stability_validation)
    stability_rows.append({
        "validation_seed": validation_seed,
        "validation_rows": len(X_stability_validation),
        "MAE": mean_absolute_error(y_stability_validation, stability_prediction),
        "RMSE": root_mean_squared_error(y_stability_validation, stability_prediction),
    })

stability_table = pd.DataFrame(stability_rows)
stability_summary = stability_table[["MAE", "RMSE"]].agg(["mean", "min", "max"])
stability_summary.loc["range"] = (
    stability_summary.loc["max"] - stability_summary.loc["min"]
)
print(f"[test를 사용하지 않은 validation seed 안정성 — Forest depth={depth_for_stability}]")
display(stability_table.round(3))
print("[seed 세 개의 평균·최솟값·최댓값·범위]")
display(stability_summary.round(3))


## 4. 최종 test 평가·오류분석·절차 검증

앞 절에서 후보와 설정 검토를 마쳤습니다. 이제 선택된 Pipeline을 전체 train에 다시 적합하고 test를 한 번만 예측합니다. 최종 지표와 절대오차 상위 관측, 시간대·근무일·날씨 조건별 오류를 확인한 뒤 실행조건을 검증합니다.


In [ ]:
# 후보와 설정 검토를 마친 뒤에만 test를 한 번 사용합니다.
final_forest_depth = globals().get("selected_forest_depth", 14)

# depth 실험에서 선택된 Random Forest 설정을 같은 fit·validation 행에서 다시 평가합니다.
tuned_forest_model = Pipeline([
    ("prep", clone(preprocess)),
    ("model", RandomForestRegressor(
        n_estimators=60,
        max_depth=final_forest_depth,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
])
tuned_forest_probe = clone(tuned_forest_model)
tuned_forest_probe.fit(X_selection_fit, y_selection_fit)
tuned_forest_fit_prediction = tuned_forest_probe.predict(X_selection_fit)
tuned_forest_validation_prediction = tuned_forest_probe.predict(X_selection_validation)
selected_forest_setting_table = pd.DataFrame([{
    "model": "random forest",
    "setting": f"max_depth={final_forest_depth}",
    "fit_MAE": mean_absolute_error(y_selection_fit, tuned_forest_fit_prediction),
    "validation_MAE": mean_absolute_error(
        y_selection_validation, tuned_forest_validation_prediction
    ),
    "fit_RMSE": root_mean_squared_error(
        y_selection_fit, tuned_forest_fit_prediction
    ),
    "validation_RMSE": root_mean_squared_error(
        y_selection_validation, tuned_forest_validation_prediction
    ),
    "fit_R2": r2_score(y_selection_fit, tuned_forest_fit_prediction),
    "validation_R2": r2_score(
        y_selection_validation, tuned_forest_validation_prediction
    ),
}])
selected_forest_setting_table[f"{PRIMARY_METRIC}_validation_minus_fit"] = (
    selected_forest_setting_table[f"validation_{PRIMARY_METRIC}"]
    - selected_forest_setting_table[f"fit_{PRIMARY_METRIC}"]
)

# 기본 Random Forest 행을 선택된 depth의 validation 지표로 갱신한 뒤 후보를 다시 정합니다.
final_candidate_validation_table = selection_table.copy()
forest_mask = final_candidate_validation_table["model"].eq("random forest")
for metric in ["MAE", "RMSE", "R2"]:
    final_candidate_validation_table.loc[forest_mask, metric] = (
        selected_forest_setting_table.loc[0, f"validation_{metric}"]
    )
final_candidate_validation_table = (
    final_candidate_validation_table.sort_values(PRIMARY_METRIC).reset_index(drop=True)
)
selected_model_name = final_candidate_validation_table.loc[0, "model"]

print("[선택된 Random Forest depth의 fit·validation 지표]")
display(selected_forest_setting_table.round(3))
print("[선택 depth를 반영해 다시 정렬한 최종 validation 후보표]")
display(final_candidate_validation_table.round(3))

if selected_model_name == "random forest":
    final_setting_model = clone(tuned_forest_model)
    final_setting = f"max_depth={final_forest_depth}"
else:
    final_setting_model = clone(models[selected_model_name])
    final_setting = "base candidate setting"

# 선택된 최종 설정을 같은 fit·validation 행에서 다시 계산합니다.
# depth가 기본값 14에서 달라졌다면 이 표와 gap도 그 깊이에 맞게 갱신됩니다.
setting_probe = clone(final_setting_model)
setting_probe.fit(X_selection_fit, y_selection_fit)
setting_fit_prediction = setting_probe.predict(X_selection_fit)
setting_validation_prediction = setting_probe.predict(X_selection_validation)
final_setting_table = pd.DataFrame([{
    "selected_model": selected_model_name,
    "setting": final_setting,
    "fit_MAE": mean_absolute_error(y_selection_fit, setting_fit_prediction),
    "validation_MAE": mean_absolute_error(
        y_selection_validation, setting_validation_prediction
    ),
    "fit_RMSE": root_mean_squared_error(y_selection_fit, setting_fit_prediction),
    "validation_RMSE": root_mean_squared_error(
        y_selection_validation, setting_validation_prediction
    ),
    "fit_R2": r2_score(y_selection_fit, setting_fit_prediction),
    "validation_R2": r2_score(
        y_selection_validation, setting_validation_prediction
    ),
}])
final_setting_table[f"{PRIMARY_METRIC}_validation_minus_fit"] = (
    final_setting_table[f"validation_{PRIMARY_METRIC}"]
    - final_setting_table[f"fit_{PRIMARY_METRIC}"]
)
selected_fit_value = final_setting_table.loc[0, f"fit_{PRIMARY_METRIC}"]
selected_validation_value = final_setting_table.loc[0, f"validation_{PRIMARY_METRIC}"]
selected_generalization_gap = selected_validation_value - selected_fit_value

print("[test를 보기 전에 확정한 최종 설정의 fit·validation 지표]")
display(final_setting_table.round(3))

final_model = clone(final_setting_model)
final_model.fit(X_train, y_train)
final_test_prediction = final_model.predict(X_test)
final_test_table = pd.DataFrame([{
    "selected_model": selected_model_name,
    "setting": final_setting,
    "test_rows": len(y_test),
    "MAE": mean_absolute_error(y_test, final_test_prediction),
    "RMSE": root_mean_squared_error(y_test, final_test_prediction),
    "R2": r2_score(y_test, final_test_prediction),
}])

print("[모든 선택을 끝낸 뒤 한 번 계산한 최종 test 결과]")
display(final_test_table.round(3))

error_table = X_test.copy()
error_table["observed"] = y_test
error_table["predicted"] = final_test_prediction
error_table["residual"] = error_table["observed"] - error_table["predicted"]
error_table["absolute_error"] = error_table["residual"].abs()
error_table["squared_error"] = error_table["residual"] ** 2
condition_columns = [column for column in [
    "hr", "workingday", "weathersit", "temp", "hum"
] if column in error_table.columns]
large_error_summary = error_table.nlargest(10, "absolute_error")[[
    "observed", "predicted", "residual", "absolute_error", *condition_columns
]].copy()
print("[최종 test에서 절대오차가 큰 관측 10건]")
display(large_error_summary.round(2))

group_columns = [column for column in ["hr", "workingday", "weathersit"] if column in error_table.columns]
condition_error_summary = (
    error_table.groupby(group_columns, observed=True)
    .agg(
        test_rows=("absolute_error", "size"),
        MAE=("absolute_error", "mean"),
        RMSE=("squared_error", lambda values: np.sqrt(values.mean())),
        mean_residual=("residual", "mean"),
    )
    .reset_index()
    .query("test_rows >= 5")
    .sort_values(PRIMARY_METRIC, ascending=False)
    .head(10)
)
print(f"[{PRIMARY_METRIC}가 큰 test 조건 — 관측 5건 이상]")
display(condition_error_summary.round(2))

checks = pd.Series({
    "four models × fit and validation": result_table.shape[0] == 8,
    "finite selection metrics": np.isfinite(
        result_table[["MAE", "RMSE", "R2"]]
    ).all().all(),
    "selection rule uses validation": selected_model_name == final_candidate_validation_table.loc[0, "model"],
    "selected forest depth metrics recorded": len(selected_forest_setting_table) == 1,
    "one selected model evaluated on test": len(final_test_table) == 1,
    "test prediction aligned": len(final_test_prediction) == len(y_test),
    "large-error summary available": len(large_error_summary) == min(10, len(y_test)),
    "final setting metrics recorded": len(final_setting_table) == 1,
    "selected fit-validation gap computed": np.isfinite(selected_generalization_gap),
}, name="passed")
display(checks.to_frame())
print(
    f"선택 후보 validation {PRIMARY_METRIC} - fit {PRIMARY_METRIC}:",
    round(float(selected_generalization_gap), 3),
)
assert checks.all()
print("모델 평가조건 검증: PASS")


## 결과 기록과 ChatGPT 검토

다음 셀은 후보표, 선택된 최종 설정의 fit·validation 지표, 최종 test 표와 큰 오류 요약을 직접 넣습니다.

- **관찰:** 기준모형 대비 주지표, 일반화 격차와 큰 오류 조건
- **조건:** 주지표 선택, 최종 모형·depth, 분할 seed
- **한계:** 미래기간·외부도시 성능과 운영비용


In [ ]:
review_prompt = f"""
Bike Sharing 모델 평가를 다음 실제 출력만 사용해 검토해 주세요.

- 사전에 정한 주지표: {PRIMARY_METRIC}
- 선택 depth를 반영한 최종 validation 후보표:
{final_candidate_validation_table.to_string(index=False)}
- 선택된 Random Forest depth의 fit·validation 지표:
{selected_forest_setting_table.to_string(index=False)}
- 선택된 최종 설정의 fit·validation 지표:
{final_setting_table.to_string(index=False)}
- 최종 test 표:
{final_test_table.to_string(index=False)}
- 큰 절대오차 관측:
{large_error_summary.to_string(index=False)}
- 필수 검사:
{checks.to_string()}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 선택 depth가 14와
다르면 final_setting_table의 해당 설정 지표와 gap만 인용하세요. validation과
test의 역할을 섞지 말고 미래기간 성능을 추측하지 마세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 완료 확인

마지막 출력이 `모델 평가조건 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 주지표를 MAE 또는 RMSE로 정한 이유와 validation에서의 기준모형 대비 차이는 무엇입니까?
- 선택모형의 fit–validation 격차와 최종 test 오류는 각각 무엇을 보여 줍니까?
- 절대오차가 큰 관측에 공통된 조건과 아직 확인하지 못한 원인을 구분했습니까?
